# Data Science for Business - Tree-based Models on Micro Mortgages

*Session 7 · ISLP 8.1 Decision Trees · 8.2.1–8.2.3 Bagging, Random Forests, Boosting · Labs 8.3.1, 8.3.3, 8.3.4*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

In India, there are about 20 million home loan (mortgage) aspirants
working in the informal sector:

- Monthly income between INR 20,000-25,000 (\$ 325-400)
- Typically no formal accounts and documents (e.g., tax returns, income proofs, bank statements)
- Often use services of money lenders with interest rates between 30 and 60% per annum

Providing mortgages to this group of customers requires to quickly and
efficiently assess their creditworthiness. Due to a lack of formal
documents and objective data, most financial institutions perform
interview-based processes to decide about these loan requests:

Strength of the current process:

-   Interview-based field assessment

-   Relaxation of document requirements

Weaknesses of the current process:

-   Costly (total transaction costs as high as 30% of loan volume)

-   Subjective judgments; depends on individual skills and motivations

-   Low reliability across branches and credit officers

-   Risk of corruption and fraud

## Load data

Load training data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_07/micromortgage.csv')

In [ ]:
data.head()

## Prepare data

Do some harmless data preparation steps: drop the ID column and turn the numeric `Tier` code into a categorical variable.

In [ ]:
data = data.drop(['ID'], axis=1)
data["Tier"] = data["Tier"].apply(lambda x: "T"+str(x))

Split data into X and y and train and test sets. About 80% of the loan requests are approved (`Decision` = 1), so we use `stratify=y` to keep this ratio identical in the training and test set.

In [ ]:
X = data.drop("Decision", axis=1)
y = data["Decision"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
y_train.value_counts(normalize=True)

Preprocess categorical features using one-hot encoding and numerical features using standard scaling. Trees split on thresholds, so scaling changes nothing for them; we keep it only to reuse the preprocessing from the previous sessions.

In [ ]:
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns
numerical_features = X_train.select_dtypes(exclude=['object', 'string']).columns

In [ ]:
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
enc.fit(X_train[categorical_features])

X_train_cat = enc.transform(X_train[categorical_features])
X_test_cat = enc.transform(X_test[categorical_features])

X_train_cat = pd.DataFrame(X_train_cat, columns=enc.get_feature_names_out(categorical_features))
X_test_cat = pd.DataFrame(X_test_cat, columns=enc.get_feature_names_out(categorical_features))

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train[numerical_features])

X_train_num = scaler.transform(X_train[numerical_features])
X_test_num = scaler.transform(X_test[numerical_features])

X_train_num = pd.DataFrame(X_train_num, columns=numerical_features)
X_test_num = pd.DataFrame(X_test_num, columns=numerical_features)

In [ ]:
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

## Classification and regression trees (CART)

First, we will grow a single classification tree (ISLP 8.1.2). The settings `criterion='entropy', max_depth=3` are exactly those of ISLP Lab 8.3.1: entropy measures node impurity, and the tree stops growing after three levels of splits.

In [ ]:
model_cart = DecisionTreeClassifier(max_depth=3, criterion='entropy', random_state=42)

In [ ]:
model_cart.fit(X_train, y_train)

Let's check how well the tree performs on the training and test data.

In [ ]:
# Training data
pred_label_train = model_cart.predict(X_train)
pred_proba_train = model_cart.predict_proba(X_train)[:,1]
acc_train = accuracy_score(y_train, pred_label_train)
auc_train = roc_auc_score(y_train, pred_proba_train)
print('ACC on training set:', round(acc_train, 2))
print('AUC on training set:', round(auc_train, 2))

print("===")

# Test data
pred_label_test = model_cart.predict(X_test)
pred_proba_test = model_cart.predict_proba(X_test)[:,1]
acc_test = accuracy_score(y_test, pred_label_test)
auc_test = roc_auc_score(y_test, pred_proba_test)
print('ACC on test set:', round(acc_test, 2))
print('AUC on test set:', round(auc_test, 2))

Keep in mind that with 80% approvals, the trivial rule "always approve" already achieves an accuracy of 0.8. This is why we look at the AUC (and later at the recall per class) as well.

The advantage of a single tree is that it is easy to interpret and visualize. Let's have a look.

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(model_cart, feature_names=model_cart.feature_names_in_, class_names=['0', '1'], filled=True, proportion=True, rounded=True, fontsize=9)
plt.show()

Read the tree top-down: each node shows the split rule, the entropy, the share of training samples in the node and the class mix (`value`). The colour shows the majority class (orange = 0 = rejected, blue = 1 = approved), and its intensity shows the purity of the node. Note that the thresholds of the numerical predictors are in standardized units (because of the scaling above).

Thanks to `max_depth=3`, this tree has at most 8 leaves and is easy to read. Without a stopping rule, the tree would keep splitting until the leaves are (almost) pure. Such a large tree is hard to interpret and overfits the training data. ISLP 8.1.1 therefore recommends growing a large tree and then **pruning** it back with cost complexity pruning. In scikit-learn, the tuning parameter α from ISLP (8.4) is called `ccp_alpha` (not to be confused with the `alpha` of the lasso).

**Your turn!** Grow a full tree (no `max_depth`) and compare its training and test AUC with the tree above. Then prune it by trying a few values of `ccp_alpha` (e.g., 0.001, 0.005, 0.01) or, like Lab 8.3.1, by tuning `ccp_alpha` with `GridSearchCV`.

In [ ]:
# YOUR CODE HERE

## Random forest

A random forest (ISLP 8.2.2) averages many deep trees grown on bootstrap samples, considering only a random subset of m predictors at each split. This decorrelates the trees and reduces the variance compared to bagging (ISLP 8.2.1), which is the special case m = p.

The number of trees B is not a critical tuning parameter: a large B does not lead to overfitting (ISLP 8.2.1), so we fix `n_estimators=500`. Instead, we tune m (`max_features`) with grid search and 5-fold cross-validation (ISLP 5.1.3): `'sqrt'` (m ≈ √p, the typical choice for classification and the scikit-learn default), a third of the predictors, and `None` (all p predictors, i.e., bagging). Since accuracy is not very informative for our imbalanced classes, we select the best model by AUC (`scoring='roc_auc'`).

In [ ]:
model_rf_tuned = GridSearchCV(
                estimator=RandomForestClassifier(n_estimators=500, random_state=42),
                param_grid={
                        'max_features': ['sqrt', 0.33, None]
                    }, scoring='roc_auc', cv=5, n_jobs=-1
                )

In [ ]:
model_rf_tuned.fit(X_train, y_train)

In [ ]:
model_rf_tuned.best_estimator_

The cross-validated AUC of each candidate value of m:

In [ ]:
pd.DataFrame(model_rf_tuned.cv_results_)[['param_max_features', 'mean_test_score', 'std_test_score']]

Make predictions on test set and evaluate the model.

In [ ]:
rf_pred_label = model_rf_tuned.predict(X_test)
rf_pred_proba = model_rf_tuned.predict_proba(X_test)[:,1]

Display precision, recall, F1-Score and accuracy.

In [ ]:
print(classification_report(y_test, rf_pred_label))

Check the recall for class 0 (rejections): it is much lower than for class 1. The forest approves most loan requests, including many that should have been rejected. The confusion matrix shows this directly (note that scikit-learn puts the true classes in the rows, while ISLP Table 4.4 puts them in the columns).

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, rf_pred_label)
plt.grid(False)
plt.show()

Plot the ROC curve and calculate the AUC.

In [ ]:
RocCurveDisplay.from_predictions(y_test, rf_pred_proba, name='Random Forest')
plt.show()

### Variable importance

A forest of 500 trees can no longer be drawn, but we can still ask which predictors matter most. Like ISLP Figure 8.9 (ISLP 8.2.1), we plot the total decrease in the Gini index (8.6) due to splits on each predictor, averaged over all trees (scikit-learn's `feature_importances_`, normalized to sum to 1).

In [ ]:
importances = pd.Series(model_rf_tuned.best_estimator_.feature_importances_, index=X_train.columns)
importances.nlargest(10).sort_values().plot.barh()
plt.xlabel('Mean decrease in Gini index')
plt.show()

## Boosted trees

Finally, we will try boosted trees (ISLP 8.2.3). Boosting grows small trees sequentially; each new tree is fit to the current residuals, and only a shrunken version of it is added to the model. We use the fast `HistGradientBoostingClassifier` (histogram-based gradient boosting), which is inspired by the LightGBM algorithm.

The three tuning parameters of boosting in ISLP map to scikit-learn as follows:

| ISLP 8.2.3 | Meaning | `HistGradientBoostingClassifier` |
|---|---|---|
| B | number of trees | `max_iter` |
| λ | shrinkage | `learning_rate` (default 0.1; ISLP uses 0.01 or 0.001) |
| d | number of splits in each tree | `max_depth=1` gives stumps (d = 1); in general `max_leaf_nodes=d+1` |

Unlike for random forests, a too large B can lead to overfitting, so we tune it together with λ and the tree depth, again with 5-fold cross-validation and AUC as the criterion.

In [ ]:
model_boost_tuned = GridSearchCV(
                estimator=HistGradientBoostingClassifier(random_state=42),
                param_grid={
                        'max_iter': [100, 500, 1000],
                        'learning_rate': [0.01, 0.1],
                        'max_depth': [1, 2, 3]
                    }, scoring='roc_auc', cv=5, n_jobs=-1
                )

In [ ]:
model_boost_tuned.fit(X_train, y_train)

In [ ]:
model_boost_tuned.best_estimator_

Make predictions on test set and evaluate the model.

In [ ]:
boost_pred_label = model_boost_tuned.predict(X_test)
boost_pred_proba = model_boost_tuned.predict_proba(X_test)[:,1]

In [ ]:
print(classification_report(y_test, boost_pred_label))

In [ ]:
RocCurveDisplay.from_predictions(y_test, boost_pred_proba, name='Gradient Boosting')
plt.show()

## Compare models

Let's put the ROC curves of the three models into one plot.

In [ ]:
fig, ax = plt.subplots()
RocCurveDisplay.from_predictions(y_test, pred_proba_test, name='CART', ax=ax)
RocCurveDisplay.from_predictions(y_test, rf_pred_proba, name='Random Forest', ax=ax)
RocCurveDisplay.from_predictions(y_test, boost_pred_proba, name='Gradient Boosting', ax=ax)
plt.show()

## Your turn!

We did only a minimal amount of hyperparameter tuning. **Your task:** Try to improve the boosted trees model by extending the search space, e.g., `learning_rate` (try 0.001), `min_samples_leaf`, `l2_regularization` or `max_leaf_nodes` instead of `max_depth`. Report the test AUC and compare it with the random forest.

In [ ]:
# YOUR CODE HERE